In [52]:
import time
import requests
import json
from pathlib import Path
import pandas as pd
from data import load_data

# ----- CONFIGURE THESE -----
data_path = r"D:\Python Projects\Hospital readmission risk\data\raw\dictionaries\unique_procedures.csv"
write_path = r"D:\Python Projects\Hospital readmission risk\data\processed\dictionaries\procedures_dictionary.csv"
procedure_sql = """
    select * 
    from `raw_data_for_dictionaries.unique_procedures`
"""
diagnoses_sql = """
    select *
    from `raw_data_for_dictionaries.unique_diagnoses`
"""
BASE = "https://termbrowser.nhs.uk/sct-browser-api/snomed"
EDITION = "uk-edition"
RELEASE = "v20251119"  # keep in sync with what you see in the browser
MAX_RETRIES = 3
BACKOFF_SECONDS = 2.0
STATE_PATH = Path("D:\Python Projects\Hospital readmission risk\data\intermediate\procedures_snomed_state.json")
CACHE: dict[str, dict] = {}
RESULTS: dict[str, dict[str, bool]] = {} 
procedure_targets = {
    "is_procedure": {
        "243796009", #situation
        "14734007", #admin_procedure
        "409073007", #education_procedure
        "108217004", #interview, history, physical examination
        "119270007", #management procedure
        "185316007", #indirect encounter
        "122869004", #measurement procedure
        "363787002", #observation
        "404684003", #clinical finding
        "72353004", #psychosocial
        "260787004", #physical_object
        "11429006" #consultation
                    },
    "is_therapy" : {"243120004"}
}    
diagnosis_targets = {
    "is_dementia" : "52448006",
    "is_cancer" : "363346000",
    "is_hiv": "86406008",
    "is_hf": "105981003",
    "is_ckd": "709044004",
    "is_diabetes": "44054006",
    "is_lf" : "235856003",
    "is_chronic": "27624003"
}
    
    
    # concept_id -> True/False results
session = requests.Session()
# ----------------------------

In [3]:
def load_state():

    global CACHE, RESULTS

    if not STATE_PATH.exists():
        return

    with STATE_PATH.open("r", encoding="utf-8") as f:
        data = json.load(f)

    CACHE = data.get("cache", {})

    raw_results = data.get("results", {})  # dict[str, dict]
    RESULTS = {
        code: {flag: bool(val) for flag, val in flags.items()}
        for code, flags in raw_results.items()
            }

In [4]:
def save_state():

    data = {
        "cache": CACHE,
        "results": RESULTS,
    }
    
    with STATE_PATH.open("w", encoding="utf-8") as f:
        json.dump(data, f)

In [5]:

def get_concept(concept_id: str) -> dict:
    """Fetch a SNOMED concept JSON once, with simple 429 handling and caching."""

    if concept_id in CACHE:

        return CACHE[concept_id]

    url = f"{BASE}/{EDITION}/{RELEASE}/concepts/{concept_id}"

    for attempt in range(1, MAX_RETRIES + 1):

        resp = session.get(url)

        if resp.status_code == 429:
            # too many requests - small backoff then retry
            time.sleep(BACKOFF_SECONDS * attempt)
            continue

        if resp.status_code in (404, 410, 500, 502, 503, 504):

            CACHE[concept_id] = {}

            for flag in procedure_targets:
                RESULTS[flag][concept_id] = False
                
            #save_state()
            return {}

        resp.raise_for_status()

        data = resp.json()

        CACHE[concept_id] = data

        #save_state()
        
        return data

    # if we get here, all retries hit 429 or other non-OK without raise
    resp.raise_for_status()

In [6]:
def get_parent_ids(concept: dict) -> list[str]:

    """Extract parent conceptIds from the concept JSON (adjust to actual structure)."""

    parents = []

    for rel in concept.get("relationships", []):
        # 'is a' relationship has conceptId 116680003 in SNOMED CT
        if rel.get("type", {}).get("conceptId") == "116680003" and rel.get("active"):

            target = rel.get("target") or {}

            cid = target.get("conceptId")

            if cid:
                
                parents.append(cid)
            
    return parents

In [44]:
def has_ancestor_in(concept_id: str, target_ids: set[str], flag: str, max_depth: int = 10) -> bool:
    # if we’ve already fully explored this concept’s ancestors in a previous run,
    # just return the previous answer if you also store it, or skip re-walk.

    if flag not in RESULTS: 
        RESULTS[flag] = {}

    if concept_id in RESULTS[flag]:
        return RESULTS[flag][concept_id]

    visited = set()
    frontier = {concept_id}
    result = False

    for depth in range(max_depth):

        next_frontier = set()

        for cid in frontier:

            if cid in visited:
                continue

            if cid in RESULTS[flag] and RESULTS[flag][cid]:

                RESULTS[flag][concept_id] = True
                #save_state()
                return True

            visited.add(cid)

            c = get_concept(cid)

            if not isinstance(c, dict):
                continue

            parents = get_parent_ids(c)

            if target_ids.intersection(parents):

                result = True
                
                RESULTS[flag][cid] = True
                RESULTS[flag][concept_id] = True
                #save_state()

                return True
            
            next_frontier.update(p for p in parents if p not in visited)

            # tiny pause to be gentle, even though this is just one chain
            time.sleep(0.1)

        if not next_frontier:
            break
        
        frontier = next_frontier

    RESULTS[flag][concept_id] = False
    #save_state()

    return result

In [28]:
def build_dictionary(data):

    for id in data['code']:

        for flag in procedure_targets:

            b = has_ancestor_in(id, procedure_targets[flag], flag)
        

In [27]:
def build_flags(data: pd.DataFrame):

    data.set_index('code', inplace = True)

    for flag in RESULTS:
    
        col = pd.Series(RESULTS[flag])
        col = col.to_frame(name = flag)
        data[flag] = col


In [10]:
def get_description(concept_id):

    if CACHE[concept_id]:

        return CACHE[concept_id].get("descriptions", []).get("term")

    return None


In [11]:
def fill_descriptions(data):

    for id in data.index:

        if data.loc[id, 'description'] is None:
            
            data.loc[id, 'description'] = get_description(id)

In [12]:
def pack_dictionary(data):

    data.to_csv(write_path)

In [50]:
if __name__ == "__main__":

    load_state()
    
    data = load_data(data_path, procedure_sql)           # load previous results

    try:
        
        build_dictionary(data)

    finally:
        save_state()

        
    build_flags(data)
    fill_descriptions(data)
    pack_dictionary(data)

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [55]:
build_flags(data)
fill_descriptions(data)

KeyError: "None of ['code'] are in the columns"

In [56]:
data

,description
code,
23426006,Measurement of respiratory function (procedure)
3802001,Dental application of desensitizing medicament...
223470000,Discussion about signs and symptoms (procedure)
58000006,Patient discharge (procedure)
287664005,Ligation of bilateral fallopian tubes (procedure)
...,...
434363004,Human epidermal growth factor receptor 2 gene ...
433114000,Human epidermal growth factor receptor 2 gene ...
105376000,Transesophageal echocardiography (procedure)


In [51]:
save_state()

In [57]:
RESULTS['is_therapy']

KeyError: 'is_therapy'

In [13]:
RESULTS

{'is_procedure': {'23426006': True,
  '3802001': False,
  '223470000': False,
  '58000006': True,
  '287664005': False,
  '363259005': True,
  '306316000': True,
  '64777005': True,
  '866146005': True,
  '399217008': False,
  '241055006': False,
  '112798008': False,
  '84100007': True,
  '67879005': True,
  '714812005': False,
  '223464006': True,
  '304531008': True,
  '418824004': False,
  '232717009': False,
  '301884003': False,
  '430925007': True,
  '104326007': True,
  '12719002': False,
  '229064008': False,
  '43075005': False,
  '129125009': True,
  '183976008': True,
  '770695002': True,
  '16335031000119103': False,
  '396487001': False,
  '410314003': True,
  '762993000': False,
  '233553003': False,
  '703423002': False,
  '315306007': True,
  '85765000': True,
  '390791001': True,
  '38102005': False,
  '88848003': False,
  '79733001': False,
  '103750000': False,
  '185087000': True,
  '709138001': True,
  '271280005': False,
  '57617002': False,
  '133899007': False,

In [15]:
for flag in RESULTS:
    
    col = pd.Series(RESULTS[flag])
    col = col.to_frame(name = flag)
    data[flag] = col

data



,description,is_procedure,is_therapy
code,,,
23426006,Measurement of respiratory function (procedure),True,False
3802001,Dental application of desensitizing medicament...,False,False
223470000,Discussion about signs and symptoms (procedure),False,False
58000006,Patient discharge (procedure),True,False
287664005,Ligation of bilateral fallopian tubes (procedure),False,False
...,...,...,...
434363004,Human epidermal growth factor receptor 2 gene ...,NaN,NaN
433114000,Human epidermal growth factor receptor 2 gene ...,NaN,NaN
105376000,Transesophageal echocardiography (procedure),NaN,NaN


In [19]:
data.iloc[203:215]

,description,is_procedure,is_therapy
code,,,
456191000124101,Postoperative care for dental procedure (regim...,NaN,NaN
47758006,Hepatitis B surface antigen measurement (proce...,NaN,NaN
252160004,Standard pregnancy test (procedure),NaN,NaN
699253003,Surgical manipulation of joint of knee (proced...,NaN,NaN
225338004,Risk assessment (procedure),NaN,NaN
410006001,Digital examination of rectum (procedure),NaN,NaN
305342007,Admission to ward (procedure),NaN,NaN
418023006,Computed tomography of chest abdomen and pelv...,NaN,NaN
370995009,Health risks education (procedure),NaN,NaN
